# RPY Shear Simulation Tutorial

This notebook demonstrates how to run Rotne–Prager–Yamakawa (RPY) Brownian dynamics simulations with shear using JAX-MD. It covers:

1. **Setting up the system**: Particle positions, box geometry, and physical parameters
2. **Configuring RPY hydrodynamics**: The Spectral-Ewald split method and parameter selection
3. **Running shear simulations**: Using `simulate.rpy_with_shear`
4. **Computing stress tensors**: Using the Irving–Kirkwood formalism
5. **Analyzing results**: Stress vs. strain and trajectory visualization

## Background

The RPY mobility tensor describes hydrodynamic interactions between spherical particles suspended in a viscous fluid. The Spectral-Ewald method splits the calculation into:
- **Real-space contributions**: Short-range, computed via neighbor lists
- **Wave-space contributions**: Long-range, computed via FFT on a grid

Under shear, the simulation box deforms over time according to a shear schedule $\gamma(t) = \dot{\gamma} \cdot t$, where $\dot{\gamma}$ is the shear rate.

In [ ]:
# Choose precision for JAX computations. 
from jax import config
config.update("jax_enable_x64", False) # Use 32-bit precision for faster computations. Set to True for higher precision if needed.

import math
import os

import jax
import jax.numpy as jnp
import numpy as np
from jax import random
import matplotlib.pyplot as plt

from jax_md import energy, minimize, partition, space, simulate, smap, util, rheo
from jax_md.hydro import rpy

print(f"JAX devices: {jax.devices()}")

## 1. System Setup

We define the basic simulation parameters:
- **Number of particles** (`N`)
- **Volume fraction** (`phi`): Determines the box size
- **Particle radius** (`a`): For RPY mobility
- **Shear rate** (`shear_rate`) or **Peclet number** (`Pe`)

### Key Parameters Explained

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| `a` | $a$ | Particle hydrodynamic radius |
| `eta` | $\eta$ | Solvent viscosity |
| `kT` | $k_B T$ | Thermal energy |
| `phi` | $\phi$ | Volume fraction $= N \cdot \frac{4\pi a^3/3}{L^3}$ |
| `xi` | $\xi$ | Ewald splitting parameter |
| `shear_rate` | $\dot{\gamma}$ | Shear rate |
| `Pe` | $\text{Pe}$ | Peclet number $= \dot{\gamma} a^2 / D_0$ |

In [ ]:
# ============================================================
# SIMULATION PARAMETERS - Modify these to your needs
# ============================================================

# System parameters
N = 64                    # Number of particles
phi = 0.2                 # Volume fraction (0.1-0.4 typical for suspensions)
dim = 3                   # Dimension (must be 3 for RPY)

# Physical parameters
a = 1.0                   # Particle radius
eta = 1.0                 # Solvent viscosity
kT = 1.0                  # Thermal energy

# Shear parameters
peclet = 1.0              # Peclet number (0 = equilibrium, >0 = sheared)
# Alternatively, set shear_rate directly:
# shear_rate = 0.1        # Direct shear rate

# Time integration
dt = 1e-4                 # Timestep (small enough for stability)
n_steps = 20000            # Total simulation steps

# RPY Ewald parameters (xi ~ 0.5/a is a good starting point)
xi = 0.5                  # Ewald splitting parameter
tol = 1e-4                # Target tolerance for RPY accuracy

# Potential parameters (soft inverse-power repulsion to prevent overlaps)
beta_eps = 146.0          # Prefactor for beta*u(r) = beta_eps * (sigma/r)^exponent
sigma = 2.0 * a           # Contact diameter (2*radius)
exponent = 170.0          # Steepness (higher = harder)
r_cut = 1.1 * sigma       # Cutoff radius

# Output control
stress_every = 10         # Record stress every N steps
traj_every = 100          # Record trajectory every N steps (0 to disable)

# Thermalization
thermalize_steps = 1000   # Equilibration steps before shear (no shear applied)

# Random seed
seed = 42

### Compute derived quantities

In [ ]:
def box_size_from_phi(n_particles, radius, phi, dim=3):
    """Compute cubic box size for given volume fraction."""
    if dim == 3:
        volume = n_particles * (4.0 / 3.0) * np.pi * radius ** 3 / phi
        return volume ** (1.0 / 3.0)
    elif dim == 2:
        area = n_particles * np.pi * radius ** 2 / phi
        return np.sqrt(area)
    raise ValueError(f"Unsupported dim={dim}")

# Box size
L = box_size_from_phi(N, a, phi, dim)
base_box = jnp.eye(dim, dtype=jnp.float64) * L

# Compute shear rate from Peclet number
D0 = kT / (6.0 * math.pi * eta * a)  # Stokes-Einstein diffusivity
shear_rate = 2.0 * peclet * D0 / (a ** 2)

print(f"Box size L = {L:.4f}")
print(f"Diffusion coefficient D0 = {D0:.4e}")
print(f"Shear rate = {shear_rate:.4e}")
print(f"Peclet number = {peclet:.2f}")
print(f"Strain per step = {shear_rate * dt:.4e}")

## 2. Define Space and Shear Schedule

The shear schedule determines how the box deforms over time. For simple shear:
$$\gamma(t) = \dot{\gamma} \cdot t$$

The `space.shearing` function returns:
- `displacement`: Computes minimum-image displacements in the deformed box
- `shift`: Moves positions respecting periodic boundaries
- `box_of`: Returns the box matrix at time $t$

In [ ]:
# Define shear schedule: gamma(t) = shear_rate * t
shear_fn = lambda t: shear_rate * t

# Create shearing space functions
# fractional_coordinates=True means positions are in [0,1]^3
# remap=True handles Lees-Edwards boundary conditions automatically
displacement, shift, box_of = space.shearing(
    base_box, 
    shear_schedule=shear_fn, 
    fractional_coordinates=True, 
    remap=True
)

# Also create static space functions for relaxation/equilibrium
displacement_0, shift_0 = space.periodic_general(base_box, fractional_coordinates=True)

print("Space functions created.")

## 3. Define the Pair Potential

We use a steep inverse-power potential to prevent particle overlaps:
$$\beta u(r) = \epsilon \left(\frac{\sigma}{r}\right)^{n}$$

This acts like a soft-sphere repulsion when particles get too close.

In [ ]:
def inverse_power(dr, epsilon, sigma, exponent, r_cut=None, r_min=1e-3, **unused):
    """Steep inverse-power pair potential."""
    r = jnp.maximum(dr, r_min)
    log_term = exponent * jnp.log(sigma / r)
    max_log = 700.0
    val = epsilon * jnp.exp(jnp.minimum(log_term, max_log))
    if r_cut is not None:
        return jnp.where(r < r_cut, val, 0.0)
    return val

# Convert to energy units
epsilon = beta_eps * kT

# Create pair energy functions
# For shearing simulation
energy_fn = smap.pair(
    inverse_power,
    space.canonicalize_displacement_or_metric(displacement),
    ignore_unused_parameters=True,
    sigma=sigma,
    epsilon=epsilon,
    exponent=exponent,
    r_cut=r_cut,
)

# For equilibrium/relaxation
energy_fn_0 = smap.pair(
    inverse_power,
    space.canonicalize_displacement_or_metric(displacement_0),
    ignore_unused_parameters=True,
    sigma=sigma,
    epsilon=epsilon,
    exponent=exponent,
    r_cut=r_cut,
)

print(f"Potential: epsilon={epsilon:.2f}, sigma={sigma:.2f}, exponent={exponent:.0f}")

## 4. Initialize and Relax Positions

We start with random positions and **must** relax them to remove overlaps before running the simulation. This is critical - random positions will have overlaps that cause numerical instability.

In [ ]:
key = random.PRNGKey(seed)
key, init_key = random.split(key)

# Random initial positions in fractional coordinates [0,1]^3
positions = random.uniform(init_key, (N, dim), dtype=jnp.float64)

print(f"Initial positions shape: {positions.shape}")
print(f"Position range: [{float(positions.min()):.3f}, {float(positions.max()):.3f}]")

In [ ]:
# CRITICAL: Relax positions to remove overlaps using neighbor list energy
# Without this step, the simulation will produce NaNs!

def relax_positions(R_init, displacement_fn, shift_fn, diameter, box, steps=500):
    """Use FIRE minimization to remove particle overlaps."""
    if steps <= 0:
        return R_init
    
    # Create soft-sphere energy with neighbor list for efficiency
    neighbor_fn, energy_relax = energy.soft_sphere_neighbor_list(
        displacement_fn,
        box,
        sigma=diameter*1.1,  # Slightly larger than diameter to properly relax overlaps
        epsilon=1.0,
        alpha=2.0,
        dr_threshold=0.2,
        fractional_coordinates=True,
        format=partition.NeighborListFormat.Sparse,
        capacity_multiplier=2.0,
    )
    neighbor = neighbor_fn.allocate(R_init, box=box)
    
    init_min, apply_min = minimize.fire_descent(energy_relax, shift_fn)
    state = init_min(R_init, neighbor=neighbor)
    
    for i in range(steps):
        neighbor = neighbor.update(state.position, box=box)
        if neighbor.did_buffer_overflow:
            neighbor = neighbor_fn.allocate(state.position, box=box)
        state = apply_min(state, neighbor=neighbor)
    
    return state.position

def count_overlaps(R, displacement_fn, diameter):
    """Count particle pairs closer than diameter."""
    # Compute pairwise displacements
    dR = space.map_product(displacement_fn)(R, R)  # (N, N, dim)
    dists = space.distance(dR)  # (N, N)
    # Upper triangle excluding diagonal (to count each pair once)
    mask = jnp.triu(jnp.ones((len(R), len(R)), dtype=bool), k=1)
    overlaps = (dists < diameter) & mask
    n_overlaps = int(jnp.sum(overlaps))
    # Min distance excluding self
    dists_no_self = jnp.where(jnp.eye(len(R), dtype=bool), jnp.inf, dists)
    min_dist = float(jnp.min(dists_no_self))
    return n_overlaps, min_dist

# Check overlaps BEFORE relaxation
n_overlaps_before, min_dist_before = count_overlaps(positions, displacement_0, sigma)
E_before = energy_fn_0(positions)
print(f"BEFORE relaxation:")
print(f"  Overlapping pairs (r < sigma): {n_overlaps_before}")
print(f"  Minimum distance: {min_dist_before:.4f} (sigma = {sigma:.2f})")
print(f"  Energy: {E_before:.4e}")

# Relax
print("\nRelaxing positions...")
positions = relax_positions(positions, displacement_0, shift_0, sigma, base_box, steps=50)

# Check overlaps AFTER relaxation
n_overlaps_after, min_dist_after = count_overlaps(positions, displacement_0, sigma)
E_after = energy_fn_0(positions)
print(f"\nAFTER relaxation:")
print(f"  Overlapping pairs (r < sigma): {n_overlaps_after}")
print(f"  Minimum distance: {min_dist_after:.4f} (sigma = {sigma:.2f})")
print(f"  Energy: {E_after:.4e}")

if n_overlaps_after > 0 or E_after > 1e6:
    print("\nWARNING: Overlaps remain after relaxation!")
    print("Try: (1) more relax steps, (2) lower phi, or (3) check sigma value")
else:
    print("\nRelaxation successful - no overlaps, ready for simulation.")

## 5. Configure RPY Parameters

The `rpy.estimate_rpy_params` function automatically selects:
- `rcut`: Real-space cutoff
- `P`: Number of Gaussian quadrature points
- `M`: FFT grid size
- `theta`: Gaussian support width

These are chosen to achieve the target tolerance `tol`.

In [ ]:
# Estimate RPY parameters
rpy_params = rpy.estimate_rpy_params(
    tol=tol,
    A=base_box,
    a=a,
    N=N,
    phi=phi,
    xi_override=xi,
    notes=True,  # Include diagnostic info
)

print("\nRPY Parameters:")
print(f"  xi (Ewald split)   = {rpy_params['xi']:.4f}")
print(f"  rcut (real-space)  = {rpy_params['rcut']:.4f}")
print(f"  P (quadrature)     = {rpy_params['P']}")
print(f"  M (FFT grid)       = {rpy_params['M']}")
print(f"  theta (Gaussian)   = {rpy_params['theta']:.4f}")

if 'notes' in rpy_params:
    notes = rpy_params['notes']
    print(f"\nError estimates:")
    print(f"  eps_r (real)  = {notes['eps_r_est']:.2e}")
    print(f"  eps_w (wave)  = {notes['eps_w_est']:.2e}")
    print(f"  eps_q (quad)  = {notes['eps_q_est']:.2e}")

## 6. Create the RPY Shear Integrator

The `simulate.rpy_with_shear` function creates:
- `init_fn`: Initialize simulation state
- `apply_fn`: Advance one timestep

In [ ]:
# Neighbor list parameters for real-space RPY
# Dense format is fastest for small systems; Sparse uses less memory for large N
Mr_params = {
    'neighbor_format': partition.NeighborListFormat.Sparse,  # or Dense
    'dr_threshold': 0.2,              # Rebuild threshold
    'capacity_multiplier': 1.5,       # Extra capacity for neighbor list
}

# Create the RPY integrator with shear
init_fn, apply_fn = simulate.rpy_with_shear(
    (displacement, shift, box_of),  # Space functions
    energy_fn,                       # Energy/force function
    dt=dt,
    kT=kT,
    a=a,
    xi=xi,
    eta=eta,
    shear_schedule=shear_fn,
    Mr_params=Mr_params,             # Neighbor list config for real-space RPY
    rcut=rpy_params['rcut'],
    P=rpy_params['P'],
    Mgrid=rpy_params['M'],
    theta=rpy_params['theta'],
)

# Create equilibrium RPY integrator (no shear) for thermalization
init_fn_eq, apply_fn_eq = simulate.rpy(
    (displacement_0, shift_0),       # Static space functions
    energy_fn_0,                     # Equilibrium energy function
    dt=dt,
    kT=kT,
    a=a,
    xi=xi,
    eta=eta,
    Mr_params=Mr_params,             # Same neighbor list config
    rcut=rpy_params['rcut'],
    P=rpy_params['P'],
    Mgrid=rpy_params['M'],
    theta=rpy_params['theta'],
)

print("RPY integrators created (shear + equilibrium).")
print(f"Real-space neighbor format: {Mr_params['neighbor_format']}")

## 7. Create Stress Tensor Function

The Irving–Kirkwood stress tensor is computed from pairwise interactions:
$$\sigma_{\alpha\beta} = -\frac{1}{V} \sum_{i<j} r_{ij,\alpha} F_{ij,\beta}$$

In [ ]:
# Create stress calculation function
stress_fn = rheo.make_pairwise_stress_fn(
    inverse_power,
    sigma=sigma,
    epsilon=epsilon,
    exponent=exponent,
    r_cut=r_cut,
)

print("Stress function created.")

## 8. Run the Simulation

First we thermalize the system (equilibrium RPY without shear), then run the shear simulation.

In [ ]:
# ============================================================
# THERMALIZATION (equilibrium, no shear)
# ============================================================
key, therm_key = random.split(key)
state_eq = init_fn_eq(therm_key, positions)

apply_fn_eq_jit = jax.jit(apply_fn_eq)

if thermalize_steps > 0:
    print(f"Thermalizing for {thermalize_steps} steps (no shear)...")
    for step in range(thermalize_steps):
        if step % 200 == 0:
            print(f"  Thermalization step {step}/{thermalize_steps}")
        
        # Check for NaNs
        if jnp.any(jnp.isnan(state_eq.mobility_position)):
            print(f"ERROR: NaN detected during thermalization at step {step}!")
            break
        
        state_eq = apply_fn_eq_jit(state_eq)
    
    # Use thermalized positions for shear run
    positions_thermalized = state_eq.mobility_position
    print(f"Thermalization complete.")
    
    # Check overlaps after thermalization
    n_overlaps_therm, min_dist_therm = count_overlaps(positions_thermalized, displacement_0, sigma)
    print(f"After thermalization: {n_overlaps_therm} overlaps, min_dist={min_dist_therm:.4f}")
else:
    positions_thermalized = positions
    print("Skipping thermalization (thermalize_steps = 0).")

In [ ]:
# ============================================================
# SHEAR SIMULATION
# ============================================================
key, run_key = random.split(key)
state = init_fn(run_key, positions_thermalized)

# Storage for results
times = []
strains = []
stresses = []
trajectories = []

# JIT compile the apply function for speed
apply_fn_jit = jax.jit(apply_fn)

print(f"Running shear simulation for {n_steps} steps...")
print(f"Total strain: {shear_rate * n_steps * dt:.4f}")

for step in range(n_steps):
    t = state.time
    gamma = shear_fn(t)
    
    # Record stress
    if step % stress_every == 0:
        current_box = box_of(t=t)
        stress_tensor = stress_fn(
            state.mobility_position,
            box=current_box,
            fractional_coordinates=True,
        )
        times.append(float(t))
        strains.append(float(gamma))
        stresses.append(np.array(stress_tensor))
    
    # Record trajectory
    if traj_every > 0 and step % traj_every == 0:
        trajectories.append({
            'time': float(t),
            'strain': float(gamma),
            'positions_frac': np.array(state.mobility_position),
            'positions_real': np.array(state.position),
        })
    
    # Progress
    if step % 500 == 0:
        print(f"  Step {step}/{n_steps}, t={float(t):.4f}, gamma={float(gamma):.4f}")
    
    # Check for NaNs
    if jnp.any(jnp.isnan(state.mobility_position)):
        print(f"ERROR: NaN detected at step {step}! Stopping.")
        break
    
    # Advance one step
    state = apply_fn_jit(state)

print("Simulation complete!")

# Convert to arrays
times = np.array(times)
strains = np.array(strains)
stresses = np.array(stresses)

## 9. Analyze Results

### 9.1 Stress vs. Strain

In [ ]:
# Extract shear stress (xy component)
sigma_xy = stresses[:, 0, 1]

# For suspensions under steady shear, the relative viscosity is:
# eta_r = sigma_xy / (eta * shear_rate)
if shear_rate > 0:
    eta_rel = sigma_xy / (eta * shear_rate)
    mean_eta_rel = np.mean(eta_rel[len(eta_rel)//2:])  # Average over second half
    print(f"Mean relative viscosity (second half): {mean_eta_rel:.4f}")

# Plot stress vs strain
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Shear stress
ax = axes[0]
ax.plot(strains, sigma_xy, 'b-', alpha=0.7)
ax.set_xlabel('Strain $\gamma$')
ax.set_ylabel('Shear stress $\sigma_{xy}$')
ax.set_title('Shear Stress vs Strain')
ax.axhline(0, color='k', linestyle='--', alpha=0.3)

# Relative viscosity
if shear_rate > 0:
    ax = axes[1]
    ax.plot(strains, eta_rel, 'r-', alpha=0.7)
    ax.axhline(mean_eta_rel, color='k', linestyle='--', label=f'Mean = {mean_eta_rel:.3f}')
    ax.set_xlabel('Strain $\gamma$')
    ax.set_ylabel('Relative viscosity $\eta_r$')
    ax.set_title('Relative Viscosity vs Strain')
    ax.legend()

plt.tight_layout()
plt.show()

### 9.2 Full Stress Tensor

In [ ]:
# Plot all stress components
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

labels = [['xx', 'xy', 'xz'], ['yx', 'yy', 'yz']]
for i in range(2):
    for j in range(3):
        ax = axes[i, j]
        component = stresses[:, i, j]
        ax.plot(strains, component, alpha=0.7)
        ax.set_xlabel('Strain $\gamma$')
        ax.set_ylabel(f'$\\sigma_{{{labels[i][j]}}}$')
        ax.set_title(f'Stress component {labels[i][j]}')
        ax.axhline(0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

### 9.3 Trajectory Visualization

In [ ]:
if len(trajectories) > 0:
    # Plot initial and final configurations
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Initial
    ax = axes[0]
    pos_init = trajectories[0]['positions_frac']
    ax.scatter(pos_init[:, 0] * L, pos_init[:, 1] * L, s=50, alpha=0.7)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Initial (t={trajectories[0]["time"]:.3f}, $\\gamma$={trajectories[0]["strain"]:.3f})')
    ax.set_xlim(0, L)
    ax.set_ylim(0, L)
    ax.set_aspect('equal')
    
    # Final
    ax = axes[1]
    pos_final = trajectories[-1]['positions_frac']
    ax.scatter(pos_final[:, 0] * L, pos_final[:, 1] * L, s=50, alpha=0.7, c='red')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Final (t={trajectories[-1]["time"]:.3f}, $\\gamma$={trajectories[-1]["strain"]:.3f})')
    ax.set_xlim(0, L)
    ax.set_ylim(0, L)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
else:
    print("No trajectories saved. Set traj_every > 0 to enable.")

### 9.4 Trajectory Movie

Animate the particle trajectories to visualize the shear flow. The box deformation under shear is also shown.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, Image, display

if len(trajectories) > 1:
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Particle scatter
    pos0 = trajectories[0]['positions_frac']
    scatter = ax.scatter(pos0[:, 0] * L, pos0[:, 1] * L, s=100, alpha=0.7, c='steelblue', edgecolors='k')
    
    # Box outline (deforms with shear)
    box_line, = ax.plot([], [], 'k-', lw=2)
    
    # Title with time and strain
    title = ax.set_title('')
    
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_xlim(-0.5 * L, 1.5 * L)
    ax.set_ylim(-0.1 * L, 1.1 * L)
    ax.set_aspect('equal')
    
    def draw_sheared_box(gamma):
        """Draw the sheared periodic box."""
        # Box corners in fractional coords: (0,0), (1,0), (1,1), (0,1)
        # Under shear: x -> x + gamma * y
        corners_x = np.array([0, L, L + gamma * L, gamma * L, 0])
        corners_y = np.array([0, 0, L, L, 0])
        return corners_x, corners_y
    
    def update(frame):
        traj = trajectories[frame]
        pos = traj['positions_frac']
        t = traj['time']
        gamma = traj['strain']
        
        # Update particle positions (apply shear to x for visualization)
        x = pos[:, 0] * L + gamma * pos[:, 1] * L
        y = pos[:, 1] * L
        scatter.set_offsets(np.column_stack([x, y]))
        
        # Update box
        bx, by = draw_sheared_box(gamma)
        box_line.set_data(bx, by)
        
        title.set_text(f't = {t:.4f}, strain $\\gamma$ = {gamma:.4f}')
        return scatter, box_line, title
    
    anim = FuncAnimation(fig, update, frames=len(trajectories), interval=100, blit=True)
    plt.close(fig)  # Prevent static display
    
    # Display animation in notebook (interactive JS player)
    display(HTML(anim.to_jshtml()))
else:
    print("Not enough trajectory frames for animation. Increase n_steps or decrease traj_every.")

In [ ]:
# Save animation as GIF and display it
gif_file = None
if len(trajectories) > 1:
    # Save as GIF (works without ffmpeg)
    try:
        gif_file = os.path.join(out_dir, 'trajectory.gif')
        anim.save(gif_file, writer='pillow', fps=10, dpi=100)
        print(f"Animation saved to {gif_file}")
    except Exception as e:
        print(f"Could not save GIF: {e}")
        gif_file = None
    
    # Also try MP4 if ffmpeg is available
    try:
        mp4_file = os.path.join(out_dir, 'trajectory.mp4')
        anim.save(mp4_file, writer='ffmpeg', fps=10, dpi=100)
        print(f"Animation also saved to {mp4_file}")
    except Exception as e:
        print(f"(MP4 not saved - ffmpeg not available)")

In [ ]:
# Display the saved GIF
if gif_file is not None and os.path.exists(gif_file):
    print(f"Displaying {gif_file}:")
    display(Image(filename=gif_file))
else:
    print("No GIF file to display.")

## 10. Advanced: Green-Kubo Viscosity Analysis

For equilibrium simulations (shear_rate = 0), you can compute the viscosity from the stress autocorrelation function using the Green-Kubo relation:

$$\eta = \frac{V}{k_B T} \int_0^\infty \langle \sigma_{xy}(t) \sigma_{xy}(0) \rangle dt$$

The `rheo` module provides tools for this analysis.

In [ ]:
# Example: stress autocorrelation (more useful for equilibrium runs)
if len(sigma_xy) > 100:
    # Compute autocorrelation
    acf = rheo.autocorrelation_fft(sigma_xy)
    time_lags = times[:len(acf)] - times[0]
    
    # Plot
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(time_lags[:len(acf)//2], acf[:len(acf)//2], 'b-')
    ax.set_xlabel('Time lag')
    ax.set_ylabel('Stress ACF')
    ax.set_title('Stress Autocorrelation Function')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data points for ACF analysis.")

## 11. Saving Results

Save the stress data to a file for later analysis.

In [ ]:
# Create output directory
out_dir = 'output/rpy_shear_tutorial'
os.makedirs(out_dir, exist_ok=True)

# Save stress data
stress_file = os.path.join(out_dir, 'stress.dat')
header = 'time strain sigma_xx sigma_xy sigma_xz sigma_yx sigma_yy sigma_yz sigma_zx sigma_zy sigma_zz'
stress_flat = stresses.reshape(len(stresses), -1)
data = np.column_stack([times, strains, stress_flat])
np.savetxt(stress_file, data, header=header, fmt='%.6e')
print(f"Stress data saved to {stress_file}")

# Save simulation parameters
params_file = os.path.join(out_dir, 'params.txt')
with open(params_file, 'w') as f:
    f.write(f"N = {N}\n")
    f.write(f"phi = {phi}\n")
    f.write(f"L = {L}\n")
    f.write(f"a = {a}\n")
    f.write(f"eta = {eta}\n")
    f.write(f"kT = {kT}\n")
    f.write(f"shear_rate = {shear_rate}\n")
    f.write(f"peclet = {peclet}\n")
    f.write(f"dt = {dt}\n")
    f.write(f"n_steps = {n_steps}\n")
    f.write(f"xi = {xi}\n")
    f.write(f"tol = {tol}\n")
    f.write(f"rpy_rcut = {rpy_params['rcut']}\n")
    f.write(f"rpy_P = {rpy_params['P']}\n")
    f.write(f"rpy_M = {rpy_params['M']}\n")
print(f"Parameters saved to {params_file}")

## Summary

This tutorial covered:

1. **System setup**: Define particles, box, and physical parameters
2. **Space functions**: `space.shearing` for Lees-Edwards boundaries
3. **Pair potential**: Soft inverse-power repulsion
4. **Position relaxation**: **Critical** - must remove overlaps before simulation
5. **RPY parameters**: `rpy.estimate_rpy_params` for automatic selection
6. **Integrator**: `simulate.rpy_with_shear` for Brownian dynamics
7. **Stress calculation**: `rheo.make_pairwise_stress_fn`
8. **Analysis**: Stress vs. strain curves and viscosity

### Key Tips

- **Timestep**: Use small `dt` (1e-4 to 1e-5) for stability
- **Relaxation**: Always relax random initial positions to remove overlaps!
- **Ewald split**: $\xi \cdot a \approx 0.5$ is a good starting point
- **Tolerance**: `tol=1e-4` is usually sufficient; lower for benchmarks
- **Peclet number**: Pe < 1 is near-equilibrium, Pe > 1 is shear-dominated
- **Volume fraction**: 0.1-0.4 for typical colloidal suspensions

### Common Issues

1. **NaN values**: Usually due to particle overlaps → increase relaxation steps
2. **Slow simulation**: Reduce N, increase tol, or use smaller M grid
3. **Unstable dynamics**: Reduce dt or shear_rate

### Next Steps

- Run multiple ensembles for better statistics
- Scan different Peclet numbers to map flow curves
- Add thermalization steps before production run
- Use neighbor lists for larger systems (see `etc/rpy_inverse_power_ensemble.py`)